![header](content/img/header.PNG)


# Part 1 : Accessing and downloading GloFAS data from CEMS Early Warning Data Store

Welcome to the companion Jupyter notebook of the 5th ICAR training -  Interactive session 1 (part 1): Accessing and downloading GloFAS data from CEMS Early Warning Data Store . This session explores the new Copernicus [Early Warning Data Store (EWDS)](https://ewds.climate.copernicus.eu).

This notebook will walk you through the basics of the new data store, inluding:
- How to create an account
- How to access data through the EWDS
- How to access data using our Python API clients
- How to use [earthkit](https://github.com/ecmwf/earthkit) to access and process the data.
- How to provide feedback and get support

## The Data Store

ECMWF operates the data store for the CEMS Early Warning Data Store:

<table border="1" style="width:100%; text-align:center;">
    <tr>
        <th>
            <h2 style="margin: 0; color: #FAA73E;">
                <a href="https://ewds.climate.copernicus.eu" target="_blank" 
                   style="text-decoration: none; color: #FAA73E;">Early Warning Data Store</a>
            </h2>
            <span style="color: #FAA73E;">Copernicus Emergency Management Service (CEMS)</span>
        </th>
    </tr>
</table>

The data stores provide free and open access to GloFAS and EFAS Data.

## Creating an account in the new Data Store

The following steps demonstrate how to register for the new Early Warning Data Store.

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
    <strong>Important</strong>: If you previously had an account on the legacy Climate or Atmosphere Data Store, your old credentials will no longer work with the new Data Stores. Instead, you will need to associate your new EWDS account with an <strong>ECMWF</strong> account - but don't worry if you don't already have one, as you can create one as part of the registration process.
</div>

If this is the first time you have visited the EWDS, you will need to register for an account by clicking "Login - Register" in the top right-hand corner.

![Login Register](content/img/login-register.PNG)

This will take you to the ECMWF login/registration page, where you will need to either sign into your existing ECMWF account, or create a new one. For more information and support about registering for an ECMWF account, see the online documentation on [getting started with ECMWF services](https://confluence.ecmwf.int/display/UDOC/Getting+started+with+ECMWF+Services+and+Systems).

Once you are logged in to your ECMWF account, you will need to complete some basic information about yourself and your interests in the Climate Data Store, along with accepting the terms of use of the data store.

## Accessing data through the EWDS

The Early Warning Data Store provides a unified user interface for accessing data, offering a simple web form where you can select the specific subset of a dataset that you would like to download.

In this example, we will download data from [Glofas forecast](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=overview) . To begin, navigate to the **[Datasets](https://ewds.climate.copernicus.eu/datasets)** page using the navigation bar at the top of the Early Warning Data Store, and search for *"glofas forecast"*. 
'River discharge and related forecasted data by Global Flood Awarness System ' will appear first in the list .

![Search Bar.PNG](content/img/ewds-search-bar.PNG)

Or you can use the filter in the top right corner to search for the dataset you're intreseted in .

![Filter.PNG](content/img/ewds-filter.PNG)

Once you have reached the *River discharge and related forecasted data by the Global Flood Awareness System* dataset, click on the title . This will take you to the download form, where you can manually select the data you are interested in.

As a simple example, let's explore today's forecast for a specific area of interest , let's say Australia and download different variables available .
Your selection should look something like this :
- System version : *Operational*
- Hydrological model : *LISFLOOD*
- Product Type : *Control forecast*
- Variable : *River discharge in the last 24 hours*
- Year : *2025*
- Month : *April*
- Day : *03*
- Leadtime hour : *24 to 720 (30 days)*

Since we're only interested in Australia, we can also use the geographical area selection tool to specify our domain of interest. Let's try going from -9°N to -39°S, and 109°W to 154°E.

![Area](content/img/area.PNG)

The last step is to select the Data format (Grib2/NetCDF-4) and Download format (Zip/Unarchived)

Once you have made your selection, accept the terms of use and click "Submit form" at the bottom of the page.

![Submit](content/img/submit-form.PNG)

This will take you to the "Your requests" page, which can also be viewed at any time by clicking *Your requests* in the top-right of the website.

The request you have just submitted will appear in your list of requests, which should look something like this:

![In progress](content/img/in-progress.PNG)

Once your request has completed, its status will change to **Complete** and you will be able to download your data by clicking the **Download** button.

![Download Completed](content/img/Download.PNG)

## Accessing data through the Python API

The Early Warning Data Store also provide programmatic access via the Python API. In this example, we will be accessing data from the EWDS with the [cdsapi Python package](https://github.com/ecmwf/cdsapi). If you get stuck while following this tutorial, you can follow more detailed instructions on this process at the [CDSAPI setup](https://ewds.climate.copernicus.eu/how-to-api) page in the EWDS.

While logged in to your EWDS account, you can visit your profile page by clicking your name in the top right of the EWDS, [or by following this link](https://ewds.climate.copernicus.eu/profile). Next, locate your **API Token** on this page, which should look something like this (please note that this is not your User ID):

![API Token ](content/img/API-Token-blur.png)

Next, you need to create a **`.cdsapirc`** file. By default, the cdsapi uses a file in your home directory (~/.cdsapirc) for credentials when connecting to the Data Stores.

The following code block will create/change this file for you. You should update `<API_TOKEN>` with your Personal Access Token from the previous step.

In [ ]:
## If you don't already have pyyaml installed, you'll need to uncomment 
## and run the pip install command below before running the next cell
# !pip install pyyaml

In [ ]:
import yaml
import os

HOME_DIR = os.path.expanduser("~")

# This is the URL for the CDS API
# If you would like to access a different data store, change this URL
URL = "https://ewds.climate.copernicus.eu/api"

# Replace <API TOKEN> with your own Personal Access Token
KEY = "<API_TOKEN>"

with open(f"{HOME_DIR}/.cdsapirc", "w") as stream:
    yaml.safe_dump(
        {"url": URL, "key": KEY},
        stream,
    )

## Make a request with the CDS API
Now that we have set up our `.cdsapirc`, we can make a request for EWDS data in a Python session. Let's make the same request we made earlier from the [River discharge and related forecasted data by Global Flood Awarness System](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=download).

First of all, if you haven't already installed the cdsapi, uncomment (remove the #) and run the following cell:

In [ ]:
#!pip install cdsapi

Navigate to the [dataset page](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=download) and fill in the form again - or alternatively, find the request you made earlier in the [Your requests](https://ewds.climate.copernicus.eu/requests?tab=all) page, then click *Details* followed by *Open request form*, which can be found next to the request ID.

Once you have completed the form, click the *Show API request code* box at the bottom of the page.
![Show-API-Request](img/Show-API-request.PNG)

This should give you the following Python code, which you can run to programmatically download the data from the EWDS:

In [1]:
import cdsapi

dataset = "cems-glofas-historical"
request = {
    "system_version": ["version_4_0"],
    "hydrological_model": ["lisflood"],
    "product_type": ["consolidated"],
    "variable": ["river_discharge_in_the_last_24_hours"],
    "hyear": ["2026"],
    "hmonth": ["01"],
    "hday": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-06-14 22:20:17,656 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).
2026-06-14 22:20:18,602 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-14 22:20:18,603 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-14 22:20:18,605 INFO Request ID is c69e5ac1-b5f5-4d7d-87d8-328cfd299b8f
2026-06-14 22:20:18,633 INFO status has been updated to accepted
2026-06-14 22:20:39,591 INFO status has been updated to running
2026-06-14 22:23:10,055 INFO status has been upda

7d243cbff0f16327397b757a86e3cae6.zip:   0%|          | 0.00/4.04M [00:00<?, ?B/s]

'7d243cbff0f16327397b757a86e3cae6.zip'

## Accessing Data Store Services data with earthkit

In the previous sections, we explored how to access data from the Copernicus Data Store Services through the **web** and through the **Python API**. In both of these cases, we downloaded the data directly onto our machine as GRIB files.

In the following section, we will use **[earthkit](https://github.com/ecmwf/earthkit)** to construct a *workflow* of data access, processing and visualisation.

**earthkit** is a Python package which provides tools for accessing, processing and visualisation earth science data. It also offers direct access to the Copernicus Data Store Services, speeding up the process of getting data from the Data Stores into your Python session.

If you do not already have **earthkit-data** and **earthkit-plots** installed, uncomment and run the following cell. We will be using **earthkit-data** to access data from the CDS, and **earthkit-plots** to visualise it.

In [ ]:
#!pip install earthkit-data
#!pip install earthkit-plots

Now we can use the `from_source()` method from **earthkit-data** to make the same request we made earlier to the CDS API.

**earthkit-data** uses the same interface as the CDS API, with one difference - the first argument to `earthkit.data.from_source()` needs to be `"cds"` if we're accessing CDS data. After that, just like the `retrieve` method from the CDS API, we need to pass the name of the dataset followed by the request details.

In [4]:
import earthkit.data

dataset = "cems-glofas-forecast"
request = {
    "system_version": ["operational"],
    "hydrological_model": ["lisflood"],
    "product_type": ["control_forecast"],
    "variable": "river_discharge_in_the_last_24_hours",
    "year": ["2025"],
    "month": ["03"],
    "day": ["19"],
    "leadtime_hour": [
        "24",
        "48",
        "72",
        "96",
        "120",
        "144",
        "168",
        "192",
        "216",
        "240",
        "264",
        "288",
        "312",
        "336",
        "360",
        "384",
        "408",
        "432",
        "456",
        "480",
        "504",
        "528",
        "552",
        "576",
        "600",
        "624",
        "648",
        "672",
        "696",
        "720"
    ],
    "data_format": "grib2",
    "download_format": "zip",
    "area": [-9, 109, -39, 154]
}


data = earthkit.data.from_source("cds", dataset, request)

2026-06-18 13:25:17,568 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-18 13:25:17,568 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-18 13:25:17,569 INFO Request ID is 18a81186-fa10-4e40-bf0e-7ee55050e1a0
2026-06-18 13:25:17,601 INFO status has been updated to accepted
2026-06-18 13:25:38,782 INFO status has been updated to running
2026-06-18 13:25:50,226 INFO status has been updated to successful


789f45cf9b6d1163e0e573f28f5eb1bc.zip:   0%|          | 0.00/4.04M [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

One advantage of using **earthkit** instead of the CDS API is that the object returned from our request can be immediately opened and explored with commonly used scientific tools like `xarray`:

In [ ]:
data.to_xarray()

## Best practices to download GloFAS Seasonal forecasts from CEMS Early Warning Data Store 

The EWDS retrieval times can vary significantly depending on the number of requests that the EWDS has at any one time and also based on the following factors that affect GloFAS:
* The priority of the dataset in question
* The size of the request
* The number of requests submitted by a user
* The number of requests to retrieve data from ECMWF Archive
* The number of requests requesting a specific dataset
* The size of the overall queue.

The EWDS delivers data as fast as possible, however, it is not an operational service and should not be relied upon to deliver data in real-time as it is produced.



In [ ]:
# === Retrieve GloFAS Historical - Morocco ===

import earthkit.data as ekd

if __name__ == '__main__':

    DATASET = 'cems-glofas-historical'
    YEARS   = ['%04d' % (y) for y in range(1979, 2027)]
    #YEARS   = ['%04d' % (y) for y in range(2026, 2027)]
    MONTHS  = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
    #MONTHS  = [ '04', '05', '06']
    DAYS    = ['%02d' % (d) for d in range(1, 32)]

    # Morocco bounding box [North, West, South, East]
    BBOX_MOROCCO = [36.2, -17.5, 20.5, -0.5]
    for year in YEARS:
        print(f'Downloading year: {year}')

        REQUEST = {
            'system_version':     'version_4_0',
            'product_type':       'consolidated',
            'hydrological_model': 'lisflood',
            'variable':           "soil_wetness_index",
            #'variable':           "river_discharge_in_the_last_24_hours",
            'hyear':              year,
            'hmonth':             MONTHS,
            'hday':               DAYS,
            'data_format':        'netcdf',
            'download_format':    'unarchived',
            'area':               BBOX_MOROCCO   # <-- added
        }

        output_path = f'/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/{DATASET}_{year}_SWI_morocco.nc'
        ds = ekd.from_source('cds', DATASET, REQUEST)
        ds.to_xarray().to_netcdf(output_path)
        print(f'Saved: {output_path}')

2026-06-19 14:51:25,189 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-19 14:51:25,190 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-19 14:51:25,191 INFO Request ID is ed4edfac-020e-46bb-b22c-aec335e31e3c
2026-06-19 14:51:25,769 INFO status has been updated to accepted
2026-06-19 14:51:59,706 INFO status has been updated to running
2026-06-19 14:54:18,760 INFO status has been updated to successful


a55ef21dfc8798a85e576553ee457b23.nc:   0%|          | 0.00/46.6M [00:00<?, ?B/s]

Saved: /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/cems-glofas-historical_1979_SWI_morocco.nc


2026-06-19 14:54:24,249 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-19 14:54:24,250 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-19 14:54:24,251 INFO Request ID is 58caa5e3-084e-481a-b411-ec7a1a6dc266
2026-06-19 14:54:24,284 INFO status has been updated to accepted
2026-06-19 14:54:37,630 INFO status has been updated to running
2026-06-19 14:57:18,091 INFO status has been updated to successful


c2b249cf4e95be573536daaae8302748.nc:   0%|          | 0.00/42.2M [00:00<?, ?B/s]

...

2026-06-19 15:16:21,690 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-19 15:16:21,690 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-19 15:16:21,691 INFO Request ID is fa2bebf9-56c5-4189-ba43-1088864426d2
2026-06-19 15:16:21,736 INFO status has been updated to accepted
2026-06-19 15:16:55,232 INFO status has been updated to running
2026-06-19 15:19:14,228 INFO status has been updated to successful


97eb5260f09033680bb69fabc08ff895.nc:   0%|          | 0.00/45.5M [00:00<?, ?B/s]

2026-06-19 15:19:18,429 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-19 15:19:18,430 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-19 15:19:18,432 INFO Request ID is bce20c15-4a29-4c78-91be-f24d256fea86


Saved: /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/cems-glofas-historical_1986_SWI_morocco.nc


2026-06-19 15:19:18,496 INFO status has been updated to accepted
2026-06-19 15:19:53,086 INFO status has been updated to running


In [20]:
import cdsapi

if __name__ == '__main__':

    DATASET = 'cems-glofas-historical'
    YEARS   = ['%04d' % (y) for y in range(2026, 2027)]
    #MONTHS  = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
    MONTHS  = [ '04', '05', '06']
    DAYS    = ['%02d' % (d) for d in range(1, 32)]

    # Morocco bounding box [North, West, South, East]
    BBOX_MOROCCO = [36.2, -17.5, 20.5, -0.5]

    for year in YEARS:
        print(f'Downloading year: {year}')

        REQUEST = {
            'system_version':     'version_4_0',
            'product_type':       'intermediate',
            'hydrological_model': 'lisflood',
            'variable':           "river_discharge_in_the_last_24_hours",
            'hyear':              year,
            'hmonth':             MONTHS,
            'hday':               DAYS,
            'data_format':        'grib2',
            'download_format':    'unarchived',
            'area':               BBOX_MOROCCO   # <-- added
        }

        output_path = f'/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/{DATASET}_{year}_intermediate_morocco.grib'

        client = cdsapi.Client()
        client.retrieve(dataset, request,output_path)

2026-06-19 14:19:17,858 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-19 14:19:17,859 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-19 14:19:17,860 INFO Request ID is 032686d5-4e10-4aa0-a470-af8c0342c140
2026-06-19 14:19:18,406 INFO status has been updated to accepted
2026-06-19 14:19:41,449 INFO status has been updated to successful


7d243cbff0f16327397b757a86e3cae6.zip:   0%|          | 0.00/4.04M [00:00<?, ?B/s]

## Combine all netcdf files

In [18]:
from pathlib import Path
import xarray as xr

data_dir = Path("/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/")

files = sorted(data_dir.glob("cems-glofas-historical_*_morocco.nc"))

ds = xr.open_mfdataset(files, combine="by_coords")

ds.to_netcdf("/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/cems-glofas-historical_morocco_1979_2026.nc")

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;"><strong>
For help concerning any problem you encounter while dealing with EWDS and CEMS data , please raise a request here</strong><a href=https://ewds.climate.copernicus.eu/help> Help and support</a></div>